# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuratulainAzhar22/flyrank-ml-work/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



The action queue ranks content using the observed CTR gap within its
search-position bucket and the amount of search volume available.

The purpose is prioritization, not automatic publishing or editing.

Higher-priority items are those with a measurable CTR gap and sufficient
impressions to make the signal more useful for human review.

Reason codes make the recommendation explainable:

- HIGH_VOLUME_CTR_GAP: high-impression content with CTR below its position benchmark.
- CTR_POSITION_GAP: lower-volume content with CTR below its position benchmark.
- NO_STRONG_SIGNAL: no strong CTR gap identified by the baseline rule.

The recommended actions are:

- PRIORITIZE_REVIEW
- REVIEW_CTR
- MONITOR

The ranking is intended to help a human reviewer decide where to investigate first.

In [2]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("My_Read_Token")

if not HF_TOKEN:
    raise ValueError("My_Read_Token is missing from Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{rel}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

print("Connection configured.")

Connection configured.


In [3]:
sample = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet('{march_path}')
    WHERE
        gsc_data_available IS TRUE
        AND gsc_impressions > 0
        AND gsc_avg_position IS NOT NULL
    LIMIT 500000
""").df()

print("Rows:", len(sample))
print("Clients:", sample["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 500000
Clients: 39


In [4]:
# Calculate CTR
sample["ctr"] = (
    sample["gsc_clicks"] / sample["gsc_impressions"]
)

# Create position buckets
sample["position_bucket"] = pd.cut(
    sample["gsc_avg_position"],
    bins=[-float("inf"), 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"]
)

# Calculate position-level CTR benchmarks
benchmarks = (
    sample.groupby("position_bucket", observed=True)
    .agg(
        total_clicks=("gsc_clicks", "sum"),
        total_impressions=("gsc_impressions", "sum")
    )
)

benchmarks["benchmark_ctr"] = (
    benchmarks["total_clicks"] /
    benchmarks["total_impressions"]
)

benchmarks = benchmarks["benchmark_ctr"]

# Add benchmark to each row
sample = sample.join(
    benchmarks,
    on="position_bucket"
)

# Calculate CTR gap
sample["ctr_gap"] = (
    sample["benchmark_ctr"] - sample["ctr"]
)

# Create score
sample["score"] = 0

sample.loc[
    (sample["gsc_impressions"] >= 100) &
    (sample["ctr"] < sample["benchmark_ctr"]),
    "score"
] = 2

sample.loc[
    (sample["gsc_impressions"] < 100) &
    (sample["ctr"] < sample["benchmark_ctr"]),
    "score"
] = 1

print(sample["score"].value_counts().sort_index())

score
0     51782
1    388867
2     59351
Name: count, dtype: int64


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the ranked action queue

action_queue = sample[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr",
        "benchmark_ctr",
        "ctr_gap",
        "score"
    ]
].copy()

# Add reason codes
action_queue["reason_code"] = "NO_STRONG_SIGNAL"

action_queue.loc[
    action_queue["score"] == 1,
    "reason_code"
] = "CTR_POSITION_GAP"

action_queue.loc[
    action_queue["score"] == 2,
    "reason_code"
] = "HIGH_VOLUME_CTR_GAP"

# Add actions
action_queue["action"] = "MONITOR"

action_queue.loc[
    action_queue["score"] == 1,
    "action"
] = "REVIEW_CTR"

action_queue.loc[
    action_queue["score"] == 2,
    "action"
] = "PRIORITIZE_REVIEW"

# Rank the queue
action_queue = action_queue.sort_values(
    ["score", "ctr_gap"],
    ascending=[False, False]
).reset_index(drop=True)

print("Rows in action queue:", len(action_queue))
print()
print(action_queue["action"].value_counts())

display(action_queue.head(20))

Rows in action queue: 500000

action
REVIEW_CTR           388867
PRIORITIZE_REVIEW     59351
MONITOR               51782
Name: count, dtype: int64


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,benchmark_ctr,ctr_gap,score,reason_code,action
0,client_73cda7b4e4f265ea,content_347a278bb1a646f5,2026-03-01,134,0,2.082090,0.0,0.004022,0.004022,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
1,client_73cda7b4e4f265ea,content_a15138d47e9949ae,2026-03-01,531,0,0.802260,0.0,0.004022,0.004022,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
2,client_73cda7b4e4f265ea,content_d8a7baed69149f40,2026-03-01,356,0,1.924157,0.0,0.004022,0.004022,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
3,client_73cda7b4e4f265ea,content_dc50704a2abee2df,2026-03-01,215,0,2.883721,0.0,0.004022,0.004022,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
4,client_73cda7b4e4f265ea,content_3ea1da426a358e8f,2026-03-01,361,0,0.429363,0.0,0.004022,0.004022,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
5,client_73cda7b4e4f265ea,content_db9fee50d56d8f1e,2026-03-01,103,0,1.155340,0.0,0.004022,0.004022,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
6,client_73cda7b4e4f265ea,content_af63c445e6d77e10,2026-03-01,134,0,0.634328,0.0,0.004022,0.004022,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
7,client_73cda7b4e4f265ea,content_ee674794b94d709f,2026-03-01,142,0,2.380282,0.0,0.004022,0.004022,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
8,client_73cda7b4e4f265ea,content_f9d3dcd14f7efb10,2026-03-01,122,0,2.573770,0.0,0.004022,0.004022,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
9,client_73cda7b4e4f265ea,content_2280349d66cb4a8a,2026-03-01,319,0,2.300940,0.0,0.004022,0.004022,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


### Intended use

The playbook is intended as a decision-support tool for prioritizing content
for human review.

A content team can use the ranked queue to identify pages with an observed
CTR gap relative to other content in the same search-position bucket.

The output should help reviewers decide where to investigate first.

### Limits

The queue does not prove that changing a page will increase CTR.

The score is based on observed data from the analysis window and should not
be interpreted as a causal recommendation.

The model should not be treated as a production automation system.

The results may change across clients, time periods, search-position
distributions, and data availability.

The recommendations should therefore be treated as directional and
decision-support signals rather than guaranteed outcomes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*



### Human review process

Every high-priority recommendation should be reviewed by a human before
any content change is made.

The reviewer should:

1. Confirm that the page has sufficient search impressions.
2. Check the current search position.
3. Inspect the page title and search snippet.
4. Check whether the search intent matches the page.
5. Check whether the content is accurate and useful.
6. Consider whether seasonality or unusual events could explain the signal.
7. Decide whether a content change is actually justified.

The queue is a prioritization aid, not an automatic editing system.

### No-go list

The system should NOT automatically:

- publish or edit content;
- change page titles or metadata;
- delete pages;
- redirect URLs;
- change canonical URLs;
- change internal links;
- launch SEO experiments;
- make claims about causality;
- spend money on content changes;
- make irreversible production changes.

Any such action requires human review and approval.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*



The queue should be monitored periodically rather than treated as permanently
valid.

A new validation or model review should be considered when:

- the input data schema changes;
- search-performance measurement changes;
- the distribution of search positions changes substantially;
- CTR distributions change substantially;
- the relationship between score and observed outcomes changes;
- a new client population is introduced;
- the underlying content mix changes;
- performance is consistently different from the validation results.

The model or scoring logic should be reconsidered rather than automatically
retrained whenever a trigger occurs.

Monitoring is intended to identify when the existing decision-support logic
may no longer represent the observed data.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Basic monitoring summary

print("Action distribution:")
print(action_queue["action"].value_counts(normalize=True).round(3))

print("\nScore distribution:")
print(action_queue["score"].value_counts(normalize=True).round(3))

print("\nCTR summary:")
print(action_queue["ctr"].describe())

print("\nPosition summary:")
print(action_queue["gsc_avg_position"].describe())

Action distribution:
action
REVIEW_CTR           0.778
PRIORITIZE_REVIEW    0.119
MONITOR              0.104
Name: proportion, dtype: float64

Score distribution:
score
1    0.778
2    0.119
0    0.104
Name: proportion, dtype: float64

CTR summary:
count    500000.000000
mean          0.003270
std           0.031292
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: ctr, dtype: float64

Position summary:
count    500000.000000
mean         13.913228
std          18.107706
min           0.000000
25%           3.500000
50%           6.836364
75%          16.238362
max         469.000000
Name: gsc_avg_position, dtype: float64


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

queue_path = output_dir / "w07_ranked_action_queue.csv"

action_queue.to_csv(
    queue_path,
    index=False
)

print("Exported:", queue_path)
print("Rows:", len(action_queue))

Exported: work/outputs/w07_ranked_action_queue.csv
Rows: 500000


**Metric Json**

In [8]:
import json

metrics = {
    "rows": int(len(action_queue)),
    "high_priority": int(
        (action_queue["action"] == "PRIORITIZE_REVIEW").sum()
    ),
    "review_ctr": int(
        (action_queue["action"] == "REVIEW_CTR").sum()
    ),
    "monitor": int(
        (action_queue["action"] == "MONITOR").sum()
    )
}

metrics_path = output_dir / "w07_action_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Metrics saved:", metrics_path)
print(metrics)

Metrics saved: work/outputs/w07_action_metrics.json
{'rows': 500000, 'high_priority': 59351, 'review_ctr': 388867, 'monitor': 51782}


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.